<a href="https://colab.research.google.com/github/grabuffo/BrainStim_ANN_fMRI_HCP/blob/main/notebooks/Test_TMS_Models_on_RestingState.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Test TMS-Trained Models on Resting-State fMRI

**Objective:** Evaluate whether models trained on TMS stimulation sessions generalize to resting-state fMRI when stimulus input is set to 0.

**Approach:**
1. Load pre-trained per-subject models (trained on TMS stimulation sessions)
2. Load resting-state fMRI data
3. Evaluate models on rest with stimulus pulse always = 0
4. Compare predictions to empirical rest activity
5. Compare performance: TMS-trained models on rest vs baseline

**Questions to answer:**
- Do TMS-trained dynamics generalize to resting state?
- How much does stimulus information help? (Train-on-stim vs train-on-stim-evaluated-without-stim)
- Are there subject-level differences in generalization?

## Step 1: Setup and Environment

In [ ]:
# --- Setup ---
# 1️⃣ Mount Google Drive (for data and trained models)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2️⃣ Clone GitHub repository (for code)
import os, sys, subprocess

repo_dir = "/content/BrainStim_ANN_fMRI_HCP"
if not os.path.exists(repo_dir):
    !git clone https://github.com/grabuffo/BrainStim_ANN_fMRI_HCP.git
else:
    print("Repo already exists ✅")

# 3️⃣ Define paths
base_dir = "/content/drive/MyDrive/Colab Notebooks/Brain_Stim_ANN/data"
data_dir = os.path.join(base_dir, "TMS_fMRI")
preproc_dir = os.path.join(base_dir, "preprocessed_subjects_tms_fmri")
dataset_pkl = os.path.join(data_dir, "dataset_tian50_schaefer400_allruns.pkl")

# Paths to previously trained models
weights_dir = os.path.join(preproc_dir, "trained_models_MLP_tms_fmri_persubject")
results_dir = os.path.join(preproc_dir, "results_tms_fmri_persubject")
test_results_dir = os.path.join(preproc_dir, "results_tms_models_on_rest")
os.makedirs(test_results_dir, exist_ok=True)

# 4️⃣ Add repo to path + imports
sys.path.append(repo_dir)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import pickle
import json
from scipy import stats
from src.preprocessing_tms_fmri import preprocess_run

import warnings
warnings.filterwarnings('ignore')

# 5️⃣ Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Step 2: Configuration and Load Dataset

In [ ]:
# --- Configuration (must match training notebook) ---
ROI_num = 450
using_steps = 3
tr_stim = 2.4
remove_initial_trs = 30
low_hz, high_hz = 0.008, 0.08

print("Configuration:")
print(f"  ROIs: {ROI_num}")
print(f"  Input dimension: {using_steps * ROI_num} (brain) + {ROI_num} (target) + 1 (stim) = {using_steps * ROI_num + ROI_num + 1}")
print(f"  Output dimension: {ROI_num}")
print(f"  Preprocessing: drop {remove_initial_trs} TRs, BP [{low_hz}, {high_hz}] Hz")

In [ ]:
# --- Load dataset ---
with open(dataset_pkl, "rb") as f:
    dataset_emp = pickle.load(f)

print(f"✅ Loaded dataset with {len(dataset_emp)} subjects")

# Check for rest data
sample_sub = next(iter(dataset_emp.keys()))
print(f"\nSample subject: {sample_sub}")
print(f"Available tasks: {list(dataset_emp[sample_sub].keys())}")

if 'task-rest' in dataset_emp[sample_sub]:
    print(f"✅ Has 'task-rest' resting-state sessions: {len(dataset_emp[sample_sub]['task-rest'])} sessions")
else:
    print("⚠️  No 'task-rest' data found!")

## Step 3: Load Pre-Trained Models

In [ ]:
# --- Define model architecture (must match training) ---
class ANN_MLP_with_Stimulus(nn.Module):
    """MLP trained on TMS data with stimulus input."""
    def __init__(self, roi_num, using_steps, hidden_dim=256, latent_dim=128):
        super().__init__()
        brain_dim = using_steps * roi_num
        stim_dim = roi_num
        stim_timing_dim = 1
        input_dim = brain_dim + stim_dim + stim_timing_dim
        output_dim = roi_num
        
        self.func = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, output_dim),
        )
    
    def forward(self, x):
        return self.func(x)

print("Model architecture defined")

In [ ]:
# --- Load trained models from training notebook ---
trained_models = {}
training_metadata = {}  # Store info about each model

# List all .pt files in the weights directory
model_files = [f for f in os.listdir(weights_dir) if f.endswith('_MLP_with_stim.pt')]
print(f"Found {len(model_files)} trained models")

for model_file in sorted(model_files):
    sub_id = model_file.replace('_MLP_with_stim.pt', '')
    model_path = os.path.join(weights_dir, model_file)
    
    # Load model
    model = ANN_MLP_with_Stimulus(roi_num=ROI_num, using_steps=using_steps)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    
    trained_models[sub_id] = model
    
    print(f"✓ {sub_id}")

print(f"\n✅ Loaded {len(trained_models)} models")

## Step 4: Evaluate TMS Models on Resting-State Data

In [ ]:
# --- Evaluate models on rest data WITHOUT stimulus (stim_pulse = 0) ---
# This tests whether the learned dynamics from TMS generalize to rest

print("="*70)
print("EVALUATING TMS-TRAINED MODELS ON RESTING-STATE DATA")
print("(Stimulus pulse always = 0)")
print("="*70)

evaluation_results = {}

for sub_id in sorted(trained_models.keys()):
    print(f"\n{sub_id}:")
    
    # Check if subject has rest data
    if 'task-rest' not in dataset_emp[sub_id]:
        print("  ⚠️  No rest data, skipping")
        continue
    
    model = trained_models[sub_id]
    model.eval()
    
    rest_sessions = dataset_emp[sub_id]['task-rest']
    sub_session_metrics = {}
    
    # --- Process each resting-state session ---
    for sess_idx in sorted(rest_sessions.keys()):
        sess_data = rest_sessions[sess_idx]
        ts = sess_data.get('time series')
        
        if ts is None or len(ts) == 0:
            continue
        
        # Preprocess (same as training)
        ts = np.asarray(ts, dtype=np.float32)
        if ts.shape[0] <= remove_initial_trs:
            continue
        
        ts_drop = ts[remove_initial_trs:, :]
        ts_proc = preprocess_run(ts_drop, tr=tr_stim, n_drop=0, 
                                 low=low_hz, high=high_hz, order=2, zscore=True)
        
        if ts_proc.shape[0] <= using_steps:
            continue
        
        T, N = ts_proc.shape
        
        # Build inputs WITHOUT stimulus (stim_pulse = 0)
        # Note: We need a stub target vector (zeros since not stimulation)
        stub_target = np.zeros(ROI_num, dtype=np.float32)  # No specific target in rest
        stim_pulse_rest = np.zeros(T, dtype=np.float32)  # No stimulus during rest
        
        rest_inputs = np.empty((T - using_steps, using_steps * N + N + 1), dtype=np.float32)
        for t in range(T - using_steps):
            brain_state = ts_proc[t : t + using_steps].reshape(-1)
            rest_inputs[t] = np.concatenate([brain_state, stub_target, [stim_pulse_rest[t]]])
        
        rest_targets = ts_proc[using_steps:, :]
        
        # Model predictions
        with torch.no_grad():
            X_rest_tensor = torch.tensor(rest_inputs, dtype=torch.float32, device=device)
            Y_pred_tensor = model(X_rest_tensor)
            Y_pred = Y_pred_tensor.cpu().numpy()
        
        # Compute metrics
        mse = np.mean((Y_pred - rest_targets) ** 2)
        
        # Per-region correlation
        corr_per_region = []
        for roi in range(N):
            if np.std(rest_targets[:, roi]) < 1e-6 or np.std(Y_pred[:, roi]) < 1e-6:
                continue
            r = np.corrcoef(rest_targets[:, roi], Y_pred[:, roi])[0, 1]
            if not np.isnan(r):
                corr_per_region.append(r)
        
        mean_corr = np.mean(corr_per_region) if corr_per_region else np.nan
        
        sub_session_metrics[sess_idx] = {
            'mse': float(mse),
            'mean_correlation': float(mean_corr),
            'n_regions': len(corr_per_region),
            'n_samples': int(rest_inputs.shape[0]),
        }
        
        print(f"  Session {sess_idx}: MSE={mse:.6f}, r={mean_corr:.3f}, n_samples={rest_inputs.shape[0]}")
    
    # Aggregate across sessions
    if sub_session_metrics:
        mses = [m['mse'] for m in sub_session_metrics.values()]
        corrs = [m['mean_correlation'] for m in sub_session_metrics.values()]
        
        evaluation_results[sub_id] = {
            'sessions': sub_session_metrics,
            'avg_mse': float(np.mean(mses)),
            'avg_correlation': float(np.nanmean(corrs)),
            'n_sessions': len(sub_session_metrics),
        }
        
        print(f"  ✓ Avg: MSE={evaluation_results[sub_id]['avg_mse']:.6f}, r={evaluation_results[sub_id]['avg_correlation']:.3f}")
    else:
        print("  No valid sessions")

print("\n" + "="*70)
print(f"Evaluated {len(evaluation_results)} subjects on resting-state")
print("="*70)

## Step 5: Compare with Training Performance

In [ ]:
# --- Load original training results for comparison ---
with open(os.path.join(results_dir, 'evaluation_results_persubject.json'), 'r') as f:
    original_eval = json.load(f)

print("Comparing TMS performance (on TMS test data) vs Rest performance:")
print("\nSubject | TMS-test r | TMS-test MSE | Rest r | Rest MSE | Gener.Gap")
print("-" * 75)

generalization_gaps = []

for sub_id in sorted(evaluation_results.keys()):
    if sub_id in original_eval:
        tms_r = original_eval[sub_id]['test']['avg_correlation']
        tms_mse = original_eval[sub_id]['test']['avg_mse']
        rest_r = evaluation_results[sub_id]['avg_correlation']
        rest_mse = evaluation_results[sub_id]['avg_mse']
        
        gap = tms_r - rest_r  # Correlation drop from TMS to Rest
        generalization_gaps.append(gap)
        
        print(f"{sub_id:12s} | {tms_r:10.3f} | {tms_mse:12.6f} | {rest_r:6.3f} | {rest_mse:8.6f} | {gap:8.3f}")

if generalization_gaps:
    print("-" * 75)
    print(f"Mean generalization gap: {np.mean(generalization_gaps):.3f}")
    print(f"  (positive = worse on rest, negative = better on rest)")

## Step 6: Visualize Results

In [ ]:
# --- Comparison plot: TMS test vs Rest performance ---
print("Generating comparison visualization...")

subjects = sorted(evaluation_results.keys())
tms_corrs = [original_eval[s]['test']['avg_correlation'] for s in subjects if s in original_eval]
rest_corrs = [evaluation_results[s]['avg_correlation'] for s in subjects]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Side-by-side comparison
x = np.arange(len(subjects))
width = 0.35

ax1.bar(x - width/2, tms_corrs, width, label='TMS Test', color='steelblue', alpha=0.8)
ax1.bar(x + width/2, rest_corrs, width, label='Rest', color='coral', alpha=0.8)

ax1.set_ylabel('Mean Correlation')
ax1.set_title('TMS-Trained Model Performance: TMS vs Rest')
ax1.set_xticks(x)
ax1.set_xticklabels(subjects, rotation=45, ha='right')
ax1.grid(True, alpha=0.3, axis='y')
ax1.legend()
ax1.axhline(np.mean(tms_corrs), color='steelblue', linestyle='--', alpha=0.5)
ax1.axhline(np.mean(rest_corrs), color='coral', linestyle='--', alpha=0.5)

# Plot 2: Generalization gap
gaps = np.array(tms_corrs) - np.array(rest_corrs)
colors = ['red' if g > 0 else 'green' for g in gaps]

ax2.bar(x, gaps, color=colors, alpha=0.7)
ax2.set_ylabel('Correlation Drop (TMS - Rest)')
ax2.set_title('Generalization Gap: TMS→Rest')
ax2.set_xticks(x)
ax2.set_xticklabels(subjects, rotation=45, ha='right')
ax2.grid(True, alpha=0.3, axis='y')
ax2.axhline(0, color='black', linestyle='-', linewidth=0.8)

plt.tight_layout()
plt.savefig(os.path.join(test_results_dir, 'tms_vs_rest_comparison.png'), dpi=150)
plt.show()

print("✓ Comparison plot saved")

## Step 7: Save Results

In [ ]:
# --- Save results ---
results_summary = {
    'method': 'TMS_trained_models_evaluated_on_rest',
    'stimulus_input': 'always 0 (no stimulus)',
    'target_input': 'stub zeros (no specific target)',
    'description': 'Evaluates whether models trained on TMS stimulation generalize to resting-state when stimulus is absent',
    'subjects': evaluation_results,
}

results_path = os.path.join(test_results_dir, 'rest_evaluation_results.json')
with open(results_path, 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"✅ Results saved: {results_path}")
print(f"\nSummary:")
print(f"  Mean correlation on rest: {np.mean(rest_corrs):.3f}")
print(f"  Mean correlation on TMS test: {np.mean(tms_corrs):.3f}")
print(f"  Mean generalization gap: {np.mean(gaps):.3f}")

## Interpretation

**Key questions answered:**

1. **Does the model generalize from TMS to rest?**
   - Compare rest correlation to TMS test correlation
   - Large gap (TMS >> Rest) → poor generalization, model learned stimulus-specific dynamics
   - Small gap → good generalization, learned general brain dynamics

2. **Which subjects generalize best?**
   - Look for subjects with small generalization gap
   - These may have learned brain dynamics independent of stimulus

3. **Does removing stimulus hurt?**
   - If rest correlation is close to TMS test → stimulus wasn't crucial
   - If rest correlation is much lower → stimulus was important for accurate predictions

**Next steps:**
- Train models on rest data directly for comparison baseline
- Train models WITHOUT stimulus channel as ablation control
- Analyze which brain regions show best/worst generalization
- Investigate why some subjects generalize better than others